# Trace and Evaluate a Chroma RAG Pipeline with Future AGI

Companion notebook for the [Chroma RAG cookbook](https://docs.futureagi.com/docs/cookbook/chroma-rag-eval).

In this notebook you will:

1. Auto-instrument **ChromaDB** and **OpenAI** with `traceAI` so every retrieval and LLM call lands as an OpenTelemetry span in your Future AGI workspace.
2. Build a small refund-policy knowledge base in an in-memory Chroma collection.
3. Run a retrieve-then-generate flow backed by `gpt-4o-mini`.
4. Score the run with five Future AGI evaluators (`context_relevance`, `chunk_attribution`, `chunk_utilization`, `completeness`, `factual_accuracy`) that diagnose retrieval and generation independently.

## Prerequisites

- Python 3.10+ (`ai-evaluation` requires it).
- A Future AGI account at [app.futureagi.com](https://app.futureagi.com) with an API key and secret key.
- An OpenAI API key.
- Estimated cost: under \$0.01 in OpenAI tokens for one query and five `turing_small` evaluator calls.

## 1. Install dependencies

In [ ]:
%pip install -q ai-evaluation traceAI-chromadb traceAI-openai chromadb openai

## 2. Set environment variables

Set `OPENAI_API_KEY`, `FI_API_KEY`, and `FI_SECRET_KEY` before running any traced code. In Colab, you can paste them into the cell below for a one-off run, but for anything beyond demo use prefer the Colab Secrets sidebar or a `.env` file.

In [ ]:
import os
from getpass import getpass

for key in ("OPENAI_API_KEY", "FI_API_KEY", "FI_SECRET_KEY"):
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

print("keys set")

## 3. Register a Future AGI tracer and instrument ChromaDB + OpenAI

After this cell, every `collection.add`, `collection.query`, and `chat.completions.create` is exported as a span to the `CHROMA_RAG_EVAL` project.

In [ ]:
from fi_instrumentation import register
from fi_instrumentation.fi_types import ProjectType
from traceai_chromadb import ChromaDBInstrumentor
from traceai_openai import OpenAIInstrumentor

trace_provider = register(
    project_type=ProjectType.OBSERVE,
    project_name="CHROMA_RAG_EVAL",
)

ChromaDBInstrumentor().instrument(tracer_provider=trace_provider)
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)

## 4. Build the Chroma knowledge base

Document four is a deliberate distractor (it has no policy information). This is what surfaces a retrieval problem later.

In [ ]:
import chromadb

client = chromadb.Client()
collection = client.create_collection("knowledge_base")

docs = [
    "Customers may request a full refund within 30 days of purchase. Refunds are processed within 5 to 7 business days after approval.",
    "To initiate a refund, contact support@example.com with your order number.",
    "Gift cards and promotional items are non-refundable.",
    "Our company was founded in 2015 and is headquartered in San Francisco.",
    "Standard shipping takes 3 to 5 business days within the US. Express shipping arrives next business day for orders placed before 12pm.",
]

collection.add(
    ids=[f"doc{i}" for i in range(len(docs))],
    documents=docs,
)
print("Collection size:", collection.count())

**First-time-it-works moment.** Open the [Future AGI dashboard](https://app.futureagi.com/dashboard/observe), select the `CHROMA_RAG_EVAL` project, and confirm a `chroma add` span has landed. If the project exists but is empty, run `trace_provider.force_flush(timeout_millis=5000)` to push pending spans.

## 5. Define the retrieve-then-generate function

Returns both the answer and the exact context strings the LLM saw, so the evaluators in Step 7 score the same retrieval the model used.

In [ ]:
from openai import OpenAI

oa = OpenAI()

def rag_answer(query: str, k: int = 3):
    hits = collection.query(query_texts=[query], n_results=k)
    contexts = hits["documents"][0]
    prompt = (
        "Use the following context to answer the question. "
        "If the context does not contain the answer, say you don't know.\n\n"
        f"Context:\n- " + "\n- ".join(contexts) + f"\n\nQuestion: {query}"
    )
    resp = oa.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content.strip(), contexts

## 6. Run a query

In [ ]:
query = "What is the refund policy and how long does processing take?"
answer, contexts = rag_answer(query)
print(answer)

Expected output (deterministic at `temperature=0`):

```
The refund policy allows customers to request a full refund within 30 days of purchase.
Refunds are processed within 5 to 7 business days after approval.
```

In the dashboard you should now see a `chroma query` span with `db.vector.query.top_k=3` and `db.vector.results.ids=["doc0","doc1","doc2"]`, followed by an OpenAI chat-completion span with prompt tokens, completion tokens, and cost.

## 7. Score the pipeline with Future AGI evaluators

Five RAG metrics, grouped by which stage they diagnose. Each is one call to `evaluate()`.

| Metric | Stage | Required keys | What it measures |
|---|---|---|---|
| `context_relevance` | Retrieval | `context`, `input` | Are the retrieved chunks relevant to the query? |
| `chunk_attribution` | Retrieval | `context`, `output` | Was the context chunk used in generating the response? |
| `chunk_utilization` | Retrieval | `context`, `output` | How effectively does the response use the context chunks? |
| `completeness` | Generation | `input`, `output` | Does the response fully address the query? |
| `factual_accuracy` | Generation | `output`, `context` (input optional) | Are the facts in the output correct? |

In [ ]:
from fi.evals import evaluate

context_str = " ".join(contexts)

scores = {}
for metric, kwargs in [
    ("context_relevance", dict(context=context_str, input=query)),
    ("chunk_attribution", dict(output=answer, context=context_str)),
    ("chunk_utilization", dict(output=answer, context=context_str)),
    ("completeness",      dict(input=query, output=answer)),
    ("factual_accuracy",  dict(input=query, output=answer, context=context_str)),
]:
    result = evaluate(metric, model="turing_small", **kwargs)
    scores[metric] = result.score
    print(f"{metric:20s} score={result.score}  passed={result.passed}")

Sample output (scores depend on the query and the corpus; ours were):

```
context_relevance    score=1.0  passed=True
chunk_attribution    score=1.0  passed=True
chunk_utilization    score=0.6  passed=True
completeness         score=1.0  passed=True
factual_accuracy     score=1.0  passed=True
```

## How to interpret the scores

| What you see | Most likely cause | Inspect first |
|---|---|---|
| Low `context_relevance`, high `chunk_attribution` | Retriever fetched off-topic chunks; the model still used them | `chroma query` span: `db.vector.results.ids` and `db.vector.results.scores` |
| High `context_relevance`, low `factual_accuracy` | Model produced claims the chunks do not support | The prompt on the OpenAI span; check for prompt drift |
| High `context_relevance`, low `chunk_utilization` | Retriever brought back too many redundant chunks | Reduce `n_results`, or add a reranker |
| `completeness` below threshold | Query asks two things; answer covers one | Split the query, or instruct the model to enumerate sub-answers |
| Everything passes but the trace is missing | Tracer registered after the first call | Move `register(...)` before any Chroma or OpenAI import that is auto-instrumented |

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ChromaDBInstrumentor().instrument(...)` raises `TypeError: wrap_function_wrapper() got an unexpected keyword argument 'module'` | `wrapt 2.x` removed the kwarg form used by the instrumentor | `%pip install "wrapt<2"` and restart the kernel |
| Banner prints but no trace shows in the dashboard | Spans were buffered when the kernel was interrupted | Call `trace_provider.force_flush(timeout_millis=15000)` |
| `evaluate(...)` raises `401 Unauthorized` | `FI_API_KEY` or `FI_SECRET_KEY` missing/invalid | Re-set them from the keys page on app.futureagi.com |
| `create_collection("kb")` raises `InvalidArgumentError` | Chroma requires collection names to be 3 to 512 alphanumeric characters | Use `knowledge_base` or any longer alphanumeric name |
| Model returns "I don't know" | Retriever ranked the company-history doc above the policy docs | Increase `k`, or inspect `chroma query`'s `db.vector.results.scores` |

## Next steps

- Replace the five-document corpus with your production knowledge base and loop the query plus `evaluate()` block over a held-out question set.
- Treat `chunk_utilization < 0.5` as a retrieval ticket rather than a generator one.
- Promote the eval block into CI: fail the pipeline if `factual_accuracy < 0.85` on the held-out set.
- Add a second retriever (e.g. a hybrid BM25 + embedding) and compare the two traces side by side in the Future AGI dashboard.
- Swap the single-shot `chat.completions.create` for a multi-turn agent and re-run this same eval block on the final answer of each conversation.